# R2_4b - Per-stage end-to-end timing

Reviewer 2.4 asks for end-to-end latency/cost. Notebook `R2_4` already
reports the per-query generation latency and cost from the master run;
this notebook adds the **per-stage** breakdown of the schema-selection
pipeline (graph build, retrieval+assembly, deterministic pruning, SQL
validation, execution) by timing the real pipeline primitives on a sample.
It is offline: cached embeddings, local SQLite execution, and generation
latency read from `generation_results.csv`. Output: `NEW_stage_timings.csv`.


In [1]:
# ===========================================================================
# PIPELINE DEFINITIONS (inlined) - from notebooks 00/01/02, DEFINITIONS ONLY.
# Self-contained: running this cell defines all graph / retrieval / context /
# scoring helpers in THIS notebook. It prepares no data, builds no contexts,
# runs no generation, and never touches generation_results.csv. Needs only
# notebooks/.env (your OpenAI key) and the kernel started in the notebooks/ dir.
# ===========================================================================

# ============================================================================
# 00_setup_and_config.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Purpose:
#   Central configuration hub imported by every other notebook. Defines project
#   paths, loads the OpenAI credentials/model tiers from notebooks/.env, sets a
#   reproducible grayscale plotting style (seaborn, dpi=600, png+pdf, no caption),
#   builds a disk-cached LLM client, and exposes shared helpers.
#
#   Anonymity note: this notebook never prints filesystem paths (absolute or
#   relative), so no local directory information can leak into shared outputs.
#   All other information (settings, model tiers, metric summaries) prints normally.
#
# Run this notebook first. Other notebooks execute it via:  %run 00_setup_and_config.ipynb
# ============================================================================


# %% [markdown]
# ## Cell 1 - Install / import dependencies
# Run once per environment. Safe to skip if already installed.

# %%
# Uncomment on first run:
# !pip install openai python-dotenv pandas numpy matplotlib seaborn sqlglot tqdm networkx scipy


# %%
# Standard library
import os
import json
import hashlib
import sqlite3
import time
from pathlib import Path

# Third party
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

print("Core libraries imported.")


# %% [markdown]
# ## Cell 2 - Project paths
# Resolve the project root RELATIVE to the notebooks/ directory so the code works
# regardless of the absolute location. Expected layout:
#
#   project_root/
#     data/
#     notebooks/          <- notebooks live here (.env is uploaded here too)
#     results/
#       figures/
#       tables/
#
# Paths are NOT printed anywhere (anonymity).

# %%
# The current working directory when a notebook runs is the notebooks/ folder.
NOTEBOOKS_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOKS_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

# Internal working directories (created if missing; not part of the required layout)
CACHE_DIR = NOTEBOOKS_DIR / ".llm_cache"        # cached LLM responses for reproducibility
CONTEXTS_DIR = RESULTS_DIR / "contexts"          # per-condition schema contexts (from notebook 01)
GENERATIONS_DIR = RESULTS_DIR / "generations"    # raw generation records (from notebook 02)

for _d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
           CACHE_DIR, CONTEXTS_DIR, GENERATIONS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Project directories ready.")


# %% [markdown]
# ## Cell 3 - Load environment variables (.env in notebooks/)
# The .env file is expected at notebooks/.env with keys:
#   OPENAI_API_KEY, LLM_MODEL_WEAK, LLM_MODEL_MID, LLM_MODEL_STRONG

# %%
ENV_PATH = NOTEBOOKS_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model tiers act as capability-level proxies for the paper's model axis:
#   STRONG -> M-L  (large API model)
#   MID    -> mid-size tier
#   WEAK   -> small-model (SLM) proxy
# NOTE: These are capability tiers, not literal parameter counts. State this in the paper.
MODEL_WEAK = os.getenv("LLM_MODEL_WEAK")
MODEL_MID = os.getenv("LLM_MODEL_MID")
MODEL_STRONG = os.getenv("LLM_MODEL_STRONG")
MODEL_REASON = os.getenv("LLM_MODEL_REASON")  # NEW (R1.2): SOTA reasoning tier (optional)

# Ordered from weakest to strongest; used by RQ4 interaction analysis.
MODEL_TIERS = {
    "WEAK": MODEL_WEAK,
    "MID": MODEL_MID,
    "STRONG": MODEL_STRONG,
    "REASON": MODEL_REASON,  # NEW (R1.2); dropped below if LLM_MODEL_REASON is unset
}
# Human-readable labels for figures (kept model-agnostic).
MODEL_TIER_LABELS = {"WEAK": "Weak", "MID": "Mid", "STRONG": "Strong",
                     "REASON": "Reasoning (SOTA)"}
MODEL_TIER_ORDER = ["WEAK", "MID", "STRONG", "REASON"]

assert OPENAI_API_KEY, "OPENAI_API_KEY not found. Place .env in the notebooks/ folder."
# REASON is optional: keep old runs working when LLM_MODEL_REASON is not set.
if not MODEL_REASON:
    MODEL_TIERS.pop("REASON"); MODEL_TIER_ORDER.remove("REASON")
    print("NOTE: LLM_MODEL_REASON unset; REASON tier skipped.")
assert all(MODEL_TIERS.values()), f"Missing model name(s) in .env: {MODEL_TIERS}"

print("Environment loaded. Model tiers:")
for k in MODEL_TIER_ORDER:
    print(f"  {k:6s} -> {MODEL_TIERS[k]}")


# %% [markdown]
# ## Cell 4 - Experiment axes (mirror the planning document)
# Conditions C1..C5, dataset ids, and metric names are declared here so every
# notebook references the same canonical strings.

# %%
# Retrieval / context conditions
CONDITIONS = {
    "C1": "Full Schema",
    "C2": "Vector-only",
    "C3": "Graph-only",
    "C4": "Hybrid",
    "C5": "Hybrid+Pruning",
}
CONDITION_ORDER = ["C1", "C2", "C3", "C4", "C5"]

# Datasets
DATASETS = {
    "D1": "Spider",
    "D2": "BIRD",
    "D2_fin": "BIRD-finance-proxy",  # domain-glossary-heavy slice for RQ5
    "D3": "In-house-finance",        # structure private; aggregate metrics only
}

# Random seed for any sampling / shuffling
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print("Conditions:", CONDITION_ORDER)
print("Datasets   :", list(DATASETS.keys()))


# %% [markdown]
# ## Cell 5 - Grayscale plotting style
# Requirements: seaborn grayscale, no caption baked into the image, dpi=600,
# save BOTH .png and .pdf. Because grayscale removes color as a channel, we also
# vary line style, marker, and hatch so series stay distinguishable in B/W print.

# %%
sns.set_theme(style="white", context="paper")

# Global rcParams for publication-grade grayscale figures.
mpl.rcParams.update({
    "figure.dpi": 120,          # on-screen preview; saved figures override to 600
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "0.85",
    "grid.linewidth": 0.5,
    "legend.frameon": False,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "image.cmap": "Greys",
    "figure.autolayout": True,
})

# Discrete grayscale palette (dark -> light) for categorical series.
def gray_palette(n):
    """Return n evenly spaced grays from dark (0.15) to light (0.75)."""
    if n <= 1:
        return ["0.2"]
    return [str(round(v, 3)) for v in np.linspace(0.15, 0.75, n)]

# Redundant encodings so grayscale series remain distinguishable in print.
LINE_STYLES = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
MARKERS = ["o", "s", "^", "D", "v", "P", "X"]
HATCHES = ["", "///", "...", "xxx", "\\\\\\", "ooo"]

# Fixed grayscale + style mapping per condition so the same condition looks the
# same across every figure in the paper.
CONDITION_STYLE = {}
_grays = gray_palette(len(CONDITION_ORDER))
for _i, _c in enumerate(CONDITION_ORDER):
    CONDITION_STYLE[_c] = {
        "color": _grays[_i],
        "linestyle": LINE_STYLES[_i % len(LINE_STYLES)],
        "marker": MARKERS[_i % len(MARKERS)],
        "hatch": HATCHES[_i % len(HATCHES)],
        "label": f"{_c} ({CONDITIONS[_c]})",
    }

print("Grayscale plotting style configured.")
print("Condition styles:", {k: v["color"] for k, v in CONDITION_STYLE.items()})


# %% [markdown]
# ## Cell 6 - Figure and table saving helpers
# Save figures to results/figures/ as BOTH png and pdf at dpi=600, no caption
# drawn on the canvas. Save tables to results/tables/ as CSV + LaTeX. Filenames
# are passed WITHOUT extension.
#
# Anonymity: these helpers confirm the save WITHOUT printing any path; they print
# only the base filename so no directory information is exposed.

# %%
def save_figure(fig, name, subdir=None):
    """
    Save a figure as {name}.png and {name}.pdf under results/figures/[subdir/].
    - dpi is enforced at 600 via rcParams savefig.dpi.
    - No caption is added to the image (per requirement).
    - Prints only the base filename (no path). Returns the list of written paths.
    """
    out_dir = FIGURES_DIR if subdir is None else (FIGURES_DIR / subdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    written = []
    for ext in ("png", "pdf"):
        path = out_dir / f"{name}.{ext}"
        fig.savefig(path, format=ext)  # dpi/bbox come from rcParams
        written.append(path)
    print(f"Saved figure: {name}.png, {name}.pdf")
    return written


def save_table(df, name, subdir=None, index=False, float_format="%.4f"):
    """
    Save a DataFrame to results/tables/ as CSV (machine-readable) and a LaTeX
    booktabs snippet (paper-ready). Filenames passed WITHOUT extension.
    Prints only the base filename (no path). Returns the CSV path.
    """
    out_dir = TABLES_DIR if subdir is None else (TABLES_DIR / subdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = out_dir / f"{name}.csv"
    tex_path = out_dir / f"{name}.tex"
    df.to_csv(csv_path, index=index)
    try:
        df.to_latex(tex_path, index=index, float_format=float_format,
                    escape=True, bold_rows=False)
    except Exception:
        pass
    print(f"Saved table: {name}.csv, {name}.tex")
    return csv_path


# %% [markdown]
# ## Cell 7 - Disk-cached OpenAI client
# Every LLM call is keyed by a hash of (model, messages, params). Identical calls
# are served from disk (reproducibility + no duplicate spend). Delete .llm_cache/
# to force fresh calls.
#
# Token-limit parameter differs by model generation: newer models (gpt-5.x)
# require 'max_completion_tokens' and reject 'max_tokens'; older models
# (gpt-4o-mini) use 'max_tokens'. We try the new name first and fall back once,
# caching which name a model accepts so we don't pay the failed call twice.

# %%
from openai import OpenAI

_client = OpenAI(api_key=OPENAI_API_KEY, timeout=90.0, max_retries=0)

# Remembers, per model, which token-limit kwarg the API accepts.
_TOKEN_PARAM_CACHE = {}


def _cache_key(model, messages, extra):
    payload = json.dumps(
        {"model": model, "messages": messages, "extra": extra},
        sort_keys=True, ensure_ascii=False,
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _create_completion(model, messages, temperature, max_tokens):
    """
    Call chat.completions handling the max_tokens vs max_completion_tokens split.
    Also handles newer models that only allow the default temperature.
    """
    # Choose the token param: use the remembered one, else try the new name first.
    token_param = _TOKEN_PARAM_CACHE.get(model, "max_completion_tokens")

    def _attempt(tp, temp):
        kwargs = {"model": model, "messages": messages, tp: max_tokens}
        if temp is not None:
            kwargs["temperature"] = temp
        return _client.chat.completions.create(**kwargs)

    for temp_try in (temperature, None):     # fall back to default temperature
        for tp_try in (token_param, "max_tokens", "max_completion_tokens"):
            try:
                resp = _attempt(tp_try, temp_try)
                _TOKEN_PARAM_CACHE[model] = tp_try   # remember what worked
                return resp
            except Exception as e:
                msg = str(e).lower()
                # If the token-param name is the problem, try the other name.
                if "max_tokens" in msg or "max_completion_tokens" in msg:
                    continue
                # If temperature is the problem, break to retry without it.
                if "temperature" in msg:
                    break
                # Any other error: re-raise (network/rate-limit handled by caller).
                raise
    # Last resort: minimal call with the new token param and no temperature.
    resp = _attempt("max_completion_tokens", None)
    _TOKEN_PARAM_CACHE[model] = "max_completion_tokens"
    return resp


def llm_complete(prompt, model=None, system=None, temperature=0.0,
                 max_tokens=1024, use_cache=True, max_retries=4):
    """
    Single-turn completion with disk caching and exponential-backoff retry.
    Returns: {text, model, prompt_tokens, completion_tokens, total_tokens,
              cached, latency_s}.
    """
    model = model or MODEL_STRONG
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    extra = {"temperature": temperature, "max_tokens": max_tokens}
    key = _cache_key(model, messages, extra)
    cache_path = CACHE_DIR / f"{key}.json"

    if use_cache and cache_path.exists():
        rec = json.loads(cache_path.read_text(encoding="utf-8"))
        rec["cached"] = True
        rec["latency_s"] = 0.0
        return rec

    last_err = None
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            resp = _create_completion(model, messages, temperature, max_tokens)
            latency = time.time() - t0
            usage = resp.usage
            rec = {
                "text": resp.choices[0].message.content or "",
                "model": model,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
                "cached": False,
                "latency_s": round(latency, 3),
            }
            cache_path.write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
            return rec
        except Exception as e:
            last_err = e
            wait = 2.0 * (2 ** attempt)
            print(f"  LLM call failed (attempt {attempt+1}/{max_retries}): {e} "
                  f"-> retry in {wait:.0f}s")
            time.sleep(wait)

    raise RuntimeError(f"LLM call failed after {max_retries} retries: {last_err}")


# %% [markdown]
# ## Cell 8 - Pricing table for cost analysis (RQ2 / Section 10)
# $ per 1M tokens. EDIT these to match current pricing for your models before
# running the cost notebook. Kept here so cost math is centralized.

# %%
# Format: model_name -> {"in": $/1M input tokens, "out": $/1M output tokens}
# Fill in the real numbers for your billing; placeholders below are examples.
PRICING_USD_PER_1M = {
    MODEL_WEAK:   {"in": 0.15, "out": 0.60},
    MODEL_MID:    {"in": 0.50, "out": 2.00},
    MODEL_STRONG: {"in": 2.50, "out": 10.00},
}
if MODEL_REASON:  # NEW (R1.2): set the reasoning tier's real $/1M before the cost notebook
    PRICING_USD_PER_1M[MODEL_REASON] = {"in": 5.00, "out": 20.00}  # EDIT to real price


def estimate_cost(model, prompt_tokens, completion_tokens):
    """Return USD cost for a single call given token counts."""
    p = PRICING_USD_PER_1M.get(model)
    if p is None or prompt_tokens is None or completion_tokens is None:
        return np.nan
    return (prompt_tokens / 1e6) * p["in"] + (completion_tokens / 1e6) * p["out"]


# %% [markdown]
# ## Cell 9 - Sanity check (optional; makes one tiny live call)
# Set RUN_LIVE_CHECK = True to verify credentials end-to-end. Costs a few tokens.

# %%
RUN_LIVE_CHECK = False

if RUN_LIVE_CHECK:
    r = llm_complete("Reply with the single word: OK",
                     model=MODEL_WEAK, max_tokens=5)
    print("Live check response:", repr(r["text"]))
    print("Tokens:", r["total_tokens"], "| cached:", r["cached"])
else:
    print("Live check skipped (set RUN_LIVE_CHECK=True to test credentials).")


# %% [markdown]
# ## Cell 10 - Persist a non-secret configuration snapshot
# Written to disk for auditability. No path is printed.

# %%
_config_snapshot = {
    "conditions": CONDITIONS,
    "datasets": DATASETS,
    "model_tiers": {k: MODEL_TIERS[k] for k in MODEL_TIER_ORDER},
    "seed": GLOBAL_SEED,
    "savefig_dpi": mpl.rcParams["savefig.dpi"],
}
(RESULTS_DIR / "config_snapshot.json").write_text(
    json.dumps(_config_snapshot, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Setup complete. Config snapshot saved.")

# ==========================================================================
# From 01_build_semantic_graph.ipynb — graph + retrieval + context assembly
# ==========================================================================

# ============================================================================
# 01_build_semantic_graph.ipynb
# ----------------------------------------------------------------------------
# Beyond Full-Schema Prompting: A Graph-based Semantic Layer for Text-to-SQL
#
# Purpose:
#   Turn each relational schema (from notebook 00b) into (a) an in-memory semantic
#   graph and (b) the per-condition schema CONTEXT strings fed to the LLM in
#   notebook 02. Builds all five conditions:
#     C1 Full Schema      - entire schema serialized (baseline)
#     C2 Vector-only      - top-k tables/columns by embedding similarity
#     C3 Graph-only       - graph traversal from full-text anchor matches
#     C4 Hybrid           - vector + full-text + graph traversal (Recall-oriented)
#     C5 Hybrid + Pruning - C4 then deterministic FK/join/column pruning (Precision)
#
#   The C4 vs C5 split is the paper's Section 8 contribution: probabilistic
#   retrieval (C4) vs deterministic pruning (C5).
#
#   SMOKE TEST: set SMOKE_TEST = True in Cell 2 to build a tiny slice first.
#   Anonymity: no filesystem paths are printed; only counts/summaries are shown.
#
# Inputs  : data/<dataset>/schema.json, data/<dataset>/questions.json
# Outputs : results/contexts/<dataset>/<condition>/<db_id>__<qid>.txt
#           results/contexts/<dataset>__context_meta.csv
#           results/contexts/schema_link_gold.csv
# ============================================================================


# %% [markdown]
# ## Cell 1 - Load shared configuration
# Pulls paths, conditions, the LLM/embedding client, and helpers from notebook 00.

# %%
# [notebook magic removed] %run 00_setup_and_config.ipynb


# %%
import re
import networkx as nx
from dataclasses import dataclass, field

print("Notebook 01 dependencies ready.")


# %% [markdown]
# ## Cell 2 - Run controls (smoke + pruning sweep + evidence-augmented retrieval)
# USE_EVIDENCE folds each question's 'evidence' hint into the retrieval query
# (embedding + anchor matching), which is critical for BIRD where column names
# are opaque. Top-k is widened to favor recall on large schemas.

# %%
SMOKE_TEST = False
SMOKE_N = 20
SMOKE_DATASETS = ["D1"]
FULL_DATASETS = ["D1", "D2", "D2_fin"]

# Retrieval hyper-parameters (widened for recall on large schemas).
K_TABLES = 8            # was 5
K_COLS = 30             # was 15
GRAPH_HOPS = 1

# Evidence-augmented retrieval (BIRD questions carry 'evidence' hints).
USE_EVIDENCE = True     # fold evidence into retrieval query + prompt context

# Pruning strength sweep.
PRUNE_LEVELS = [1, 2, 3]
PRUNE_KEEP_TOP_COLS = 12       # was 8; wider embedding-preserve at level 2
C5_CONDITIONS = {f"C5_L{lv}": lv for lv in PRUNE_LEVELS}
R2_1_CONDITIONS = ["EV00", "EV10", "EV01", "EV11", "C1_EV0", "C1_EV1",
                   "ABL_full", "ABL_nograph", "ABL_nogloss", "ABL_noval", "ABL_nojoin"]  # REVISION R2.1
BUILD_CONDITIONS = ["C1", "C2", "C3", "C4"] + list(C5_CONDITIONS.keys()) + R2_1_CONDITIONS

if SMOKE_TEST:
    DATASETS_TO_BUILD = SMOKE_DATASETS
    BUILD_LIMIT = SMOKE_N
else:
    DATASETS_TO_BUILD = FULL_DATASETS
    BUILD_LIMIT = None

print(f"Mode: {'SMOKE' if SMOKE_TEST else 'FULL'} | datasets={DATASETS_TO_BUILD} "
      f"| limit={BUILD_LIMIT} | evidence={USE_EVIDENCE} "
      f"| k_tables={K_TABLES} k_cols={K_COLS}")


# %% [markdown]
# ## Cell 3 - Data structures for the semantic graph

# %%
@dataclass
class ColumnNode:
    table: str
    name: str
    dtype: str
    pk: bool = False
    desc: str = ""
    sample_values: list = field(default_factory=list)

    @property
    def qualified(self):
        return f"{self.table}.{self.name}"


@dataclass
class SchemaGraph:
    db_id: str
    graph: nx.MultiDiGraph
    columns: dict          # qualified name -> ColumnNode
    tables: list           # table names
    glossary: list         # glossary dicts
    join_patterns: list    # join-pattern dicts

    def table_columns(self, table):
        return [c for c in self.columns.values() if c.table == table]


# %% [markdown]
# ## Cell 4 - Build the semantic graph from a schema.json entry
# Nodes: Table, Column, Concept (glossary). Edges: HAS_COLUMN, FK, FK_TABLE, MAPS_TO.

# %%
def _load_schema_list(dataset_key):
    """Load and cache the full schema.json (a list of db dicts) for a dataset."""
    path = DATA_DIR / dataset_key / "schema.json"
    return json.loads(path.read_text(encoding="utf-8"))


def build_graph(schema_entry):
    """Construct a SchemaGraph from one schema.json db entry."""
    db_id = schema_entry["db_id"]
    G = nx.MultiDiGraph()
    columns = {}
    tables = list(schema_entry["tables"].keys())

    # Table + Column nodes
    for tname, tinfo in schema_entry["tables"].items():
        G.add_node(("table", tname), ntype="table", name=tname)
        for cname, cinfo in tinfo["columns"].items():
            col = ColumnNode(
                table=tname, name=cname,
                dtype=cinfo.get("type", "TEXT"),
                pk=bool(cinfo.get("pk", False)),
                desc=cinfo.get("desc", ""),
                sample_values=cinfo.get("sample_values", []),
            )
            columns[col.qualified] = col
            G.add_node(("column", col.qualified), ntype="column",
                       name=col.qualified, dtype=col.dtype, desc=col.desc)
            G.add_edge(("table", tname), ("column", col.qualified), etype="HAS_COLUMN")

    # Foreign-key edges (both directions for traversal convenience)
    for fk in schema_entry.get("foreign_keys", []):
        src, dst = fk["from"], fk["to"]
        if ("column", src) in G and ("column", dst) in G:
            G.add_edge(("column", src), ("column", dst), etype="FK")
            G.add_edge(("column", dst), ("column", src), etype="FK")
            st, dt = src.split(".")[0], dst.split(".")[0]
            G.add_edge(("table", st), ("table", dt), etype="FK_TABLE", on=f"{src}={dst}")
            G.add_edge(("table", dt), ("table", st), etype="FK_TABLE", on=f"{src}={dst}")

    # Concept (glossary) nodes + MAPS_TO edges
    for g in schema_entry.get("glossary", []):
        cid = ("concept", g["term"])
        G.add_node(cid, ntype="concept", name=g["term"],
                   definition=g.get("definition", ""))
        for target in g.get("maps_to", []):
            node = ("column", target) if target in columns else ("table", target)
            if node in G:
                G.add_edge(cid, node, etype="MAPS_TO")

    return SchemaGraph(
        db_id=db_id, graph=G, columns=columns, tables=tables,
        glossary=schema_entry.get("glossary", []),
        join_patterns=schema_entry.get("join_patterns", []),
    )


# %% [markdown]
# ## Cell 5 - Embeddings for vector retrieval (C2/C4/C5)
# Cached OpenAI embeddings. Each table/column is embedded once from a short
# natural-language descriptor (name + description + sample values).

# %%
EMBED_MODEL = "text-embedding-3-small"
EMBED_CACHE = CACHE_DIR / "embeddings"
EMBED_CACHE.mkdir(parents=True, exist_ok=True)


def _embed_cache_path(text):
    key = hashlib.sha256((EMBED_MODEL + "||" + text).encode("utf-8")).hexdigest()
    return EMBED_CACHE / f"{key}.json"


def embed_text(text):
    """Return a cached embedding vector for a single string."""
    text = text[:8000]   # REVISION: guard the embedding model's 8192-token input limit
    p = _embed_cache_path(text)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    resp = _client.embeddings.create(model=EMBED_MODEL, input=text)
    vec = resp.data[0].embedding
    p.write_text(json.dumps(vec), encoding="utf-8")
    return vec


def _descriptor_for_column(col: ColumnNode):
    sv = ", ".join(str(v) for v in col.sample_values[:5])
    return f"table {col.table} column {col.name} ({col.dtype}). {col.desc}. examples: {sv}"


def _descriptor_for_table(sg: SchemaGraph, table):
    cols = ", ".join(c.name for c in sg.table_columns(table))
    return f"table {table} with columns: {cols}"


def cosine(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12
    return float(np.dot(a, b) / denom)


# %% [markdown]
# ## Cell 6 - Retrieval primitives
# Full-text anchor matching, vector top-k, graph expansion, shortest join path.

# %%
def anchor_match(sg: SchemaGraph, question):
    """Deterministic full-text matches (tables/cols/glossary terms in the question)."""
    q = question.lower()
    q_tokens = set(re.findall(r"[a-z_]+", q))
    matched_tables, matched_cols = set(), set()

    for t in sg.tables:
        if t.lower() in q or t.lower().replace("_", " ") in q:
            matched_tables.add(t)
    for qn, col in sg.columns.items():
        name_tok = col.name.lower()
        if name_tok in q_tokens or name_tok.replace("_", " ") in q:
            matched_cols.add(qn)
            matched_tables.add(col.table)
    for g in sg.glossary:
        if g["term"].lower() in q:
            for target in g.get("maps_to", []):
                if target in sg.columns:
                    matched_cols.add(target)
                    matched_tables.add(target.split(".")[0])
                elif target in sg.tables:
                    matched_tables.add(target)
    return matched_tables, matched_cols


def vector_topk(sg: SchemaGraph, question, k_tables=K_TABLES, k_cols=K_COLS):
    """Embedding similarity retrieval over tables and columns."""
    qv = embed_text(question)

    tbl_scores = [(t, cosine(qv, embed_text(_descriptor_for_table(sg, t))))
                  for t in sg.tables]
    tbl_scores.sort(key=lambda x: x[1], reverse=True)
    top_tables = [t for t, _ in tbl_scores[:k_tables]]

    col_scores = [(qn, cosine(qv, embed_text(_descriptor_for_column(col))))
                  for qn, col in sg.columns.items()]
    col_scores.sort(key=lambda x: x[1], reverse=True)
    top_cols = [c for c, _ in col_scores[:k_cols]]

    tables = set(top_tables) | {c.split(".")[0] for c in top_cols}
    return tables, set(top_cols)


def graph_expand(sg: SchemaGraph, seed_tables, hops=GRAPH_HOPS):
    """Expand seed tables along FK_TABLE edges up to `hops`."""
    frontier = set(seed_tables)
    visited = set(seed_tables)
    for _ in range(hops):
        nxt = set()
        for t in frontier:
            node = ("table", t)
            if node not in sg.graph:
                continue
            for _, dst, data in sg.graph.out_edges(node, data=True):
                if data.get("etype") == "FK_TABLE" and dst[1] not in visited:
                    nxt.add(dst[1])
        visited |= nxt
        frontier = nxt
    return visited


def shortest_join_path(sg: SchemaGraph, tables):
    """Return (kept_tables, join_conditions) on shortest FK paths between tables."""
    tables = list(tables)
    join_conds, keep_tables = set(), set(tables)
    tbl_graph = nx.Graph()
    for u, v, data in sg.graph.edges(data=True):
        if data.get("etype") == "FK_TABLE":
            tbl_graph.add_edge(u[1], v[1], on=data.get("on", ""))
    for i in range(len(tables)):
        for j in range(i + 1, len(tables)):
            a, b = tables[i], tables[j]
            if a in tbl_graph and b in tbl_graph and nx.has_path(tbl_graph, a, b):
                path = nx.shortest_path(tbl_graph, a, b)
                keep_tables.update(path)
                for x, y in zip(path[:-1], path[1:]):
                    on = tbl_graph[x][y].get("on", "")
                    if on:
                        join_conds.add(on)
    return keep_tables, sorted(join_conds)


# %% [markdown]
# ## Cell 7 - Deterministic pruning with adjustable strength (paper Section 8.2)
# level 1: drop FK-isolated tables only (columns untouched).
# level 2: + minimize to join-path tables; project columns but PRESERVE the top
#          embedding-similar columns so semantically-relevant gold cols survive.
# level 3: aggressive - strict token/type projection (original C5 behavior).
# Uses graph facts + question tokens; level 2 also uses cached embeddings.

# %%
def extract_literals(question):
    strings = re.findall(r"'([^']+)'|\"([^\"]+)\"", question)
    strings = [s1 or s2 for s1, s2 in strings]
    numbers = re.findall(r"\b\d+(?:\.\d+)?\b", question)
    return strings, numbers


def _is_fk_column(sg: SchemaGraph, qualified):
    node = ("column", qualified)
    if node not in sg.graph:
        return False
    for _, _, d in sg.graph.out_edges(node, data=True):
        if d.get("etype") == "FK":
            return True
    for _, _, d in sg.graph.in_edges(node, data=True):
        if d.get("etype") == "FK":
            return True
    return False


def _embedding_top_cols(sg: SchemaGraph, question, candidate_cols, top_k):
    """Return the top_k candidate columns by embedding similarity to the question."""
    if not candidate_cols or top_k <= 0:
        return set()
    qv = embed_text(question)
    scored = []
    for qn in candidate_cols:
        col = sg.columns.get(qn)
        if col is None:
            continue
        scored.append((qn, cosine(qv, embed_text(_descriptor_for_column(col)))))
    scored.sort(key=lambda x: x[1], reverse=True)
    return {qn for qn, _ in scored[:top_k]}


def _fk_connected_tables(sg: SchemaGraph, cand_tables):
    """Candidate tables reachable from an anchor via FK paths (drops only
    FK-isolated candidates). Helper for the monotone pruning below."""
    keep, _ = shortest_join_path(sg, set(cand_tables))
    keep = keep & set(cand_tables)
    return keep if keep else set(cand_tables)


def deterministic_prune(sg: SchemaGraph, cand_tables, cand_cols, question,
                        level=3, keep_top_cols=PRUNE_KEEP_TOP_COLS):
    """
    REVISION (R2.3-ii): MONOTONE deterministic pruning. Each level is a strict
    subset of the previous one -- C4 ⊇ L1 ⊇ L2 ⊇ L3 in both tables and columns --
    so strict column recall is non-increasing with `level` on every dataset.
    Pruning only REMOVES from the C4 candidate set (cand_tables, cand_cols);
    nothing is ever re-added (this is what the old L2 did, breaking monotonicity).
    Returns (kept_tables, kept_cols, join_conds).
    """
    q = question.lower()
    q_tokens = set(re.findall(r"[a-z_]+", q))
    strings, numbers = extract_literals(question)
    anchor_tables, anchor_cols = anchor_match(sg, question)

    # Universe = the C4 candidate set; columns restricted to candidate tables.
    U_tables = set(cand_tables)
    U_cols = {qn for qn in cand_cols if qn.split(".")[0] in U_tables}

    # ---- L1: drop only FK-isolated tables; keep their candidate columns ----
    l1_tables = _fk_connected_tables(sg, U_tables)
    l1_cols = {qn for qn in U_cols if qn.split(".")[0] in l1_tables}
    _, l1_joins = shortest_join_path(sg, l1_tables)
    if level <= 1:
        return l1_tables, l1_cols, l1_joins

    # ---- L2: restrict to join-path tables (⊆ L1); project columns (⊆ L1) ----
    seed = (anchor_tables & l1_tables) or l1_tables
    jp_tables, l2_joins = shortest_join_path(sg, seed)
    l2_tables = (jp_tables & l1_tables) or l1_tables
    l2_pool = {qn for qn in l1_cols if qn.split(".")[0] in l2_tables}
    emb_keep = _embedding_top_cols(sg, question, list(l2_pool), keep_top_cols)
    l2_cols = set()
    for qn in l2_pool:
        col = sg.columns[qn]
        name_tok = col.name.lower()
        if (qn in anchor_cols or col.pk or _is_fk_column(sg, qn)
                or name_tok in q_tokens or name_tok.replace("_", " ") in q
                or qn in emb_keep):
            l2_cols.add(qn)
    # keys of kept tables, but only if already present in L1 (preserve nesting)
    for t in l2_tables:
        for col in sg.table_columns(t):
            if (col.pk or _is_fk_column(sg, col.qualified)) and col.qualified in l1_cols:
                l2_cols.add(col.qualified)
    l2_cols &= l1_cols                      # enforce L2 ⊆ L1
    if level == 2:
        return l2_tables, l2_cols, l2_joins

    # ---- L3: strict token/type projection within L2 ----
    l3_cols = set(l2_cols)
    if numbers and not strings:
        drop = {qn for qn in l3_cols
                if sg.columns[qn].dtype.upper() in {"TEXT", "VARCHAR", "CHAR"}
                and qn not in anchor_cols
                and not sg.columns[qn].pk and not _is_fk_column(sg, qn)}
        l3_cols -= drop
    l3_cols &= l2_cols                      # enforce L3 ⊆ L2
    return set(l2_tables), l3_cols, l2_joins

# %% [markdown]
# ## Cell 8 - Serialize a schema subset into an LLM context string

# %%
def render_context(sg: SchemaGraph, tables, cols, join_conds=None,
                   include_glossary=True, include_samples=True, include_joins=True):
    """Serialize a schema subset to a stable string for the LLM prompt."""
    tables = sorted(tables)
    lines = [f"-- Database: {sg.db_id}", "-- Schema (relevant subset):"]

    for t in tables:
        lines.append(f"\nTABLE {t} (")
        tcols = [c for c in sg.table_columns(t)
                 if (cols is None or c.qualified in cols or c.pk)]
        if not tcols:
            tcols = [c for c in sg.table_columns(t) if c.pk] or sg.table_columns(t)[:1]
        for c in tcols:
            tag = " PRIMARY KEY" if c.pk else ""
            desc = f"  -- {c.desc}" if c.desc else ""
            samples = ""
            if include_samples and c.sample_values:
                sv = ", ".join(str(v) for v in c.sample_values[:3])
                samples = f" [e.g., {sv}]"
            lines.append(f"    {c.name} {c.dtype}{tag},{desc}{samples}")
        lines.append(")")

    if include_glossary and sg.glossary:
        relevant = [g for g in sg.glossary
                    if any(m.split('.')[0] in tables for m in g.get("maps_to", []))]
        if relevant:
            lines.append("\n-- Business glossary:")
            for g in relevant:
                lines.append(f"--   {g['term']}: {g.get('definition','')} "
                             f"-> {', '.join(g.get('maps_to', []))}")

    if include_joins and join_conds:
        lines.append("\n-- Suggested join paths:")
        for j in join_conds:
            lines.append(f"--   {j}")

    return "\n".join(lines)


# %% [markdown]
# ## Cell 9 - Per-condition context builder (evidence-augmented)
# The retrieval query is question (+ evidence when USE_EVIDENCE). Evidence is also
# appended to the rendered context so the generator can use the same hint.

# %%
def _retrieval_query(question, evidence):
    if USE_EVIDENCE and evidence:
        return f"{question}\nHint: {evidence}"
    return question


def anchor_match_nogloss(sg: SchemaGraph, question):
    """REVISION (R2.1 ablation): anchor_match WITHOUT the business glossary,
    to measure the glossary's independent contribution."""
    q = question.lower()
    q_tokens = set(re.findall(r"[a-z_]+", q))
    matched_tables, matched_cols = set(), set()
    for t in sg.tables:
        if t.lower() in q or t.lower().replace("_", " ") in q:
            matched_tables.add(t)
    for qn, col in sg.columns.items():
        name_tok = col.name.lower()
        if name_tok in q_tokens or name_tok.replace("_", " ") in q:
            matched_cols.add(qn); matched_tables.add(col.table)
    return matched_tables, matched_cols


# REVISION (R2.1): evidence 2x2 (retrieval vs generation channel) + metadata
# leave-one-out. ev_ret = evidence in the retrieval query; ev_prompt = evidence
# in the generation prompt; graph/glossary/values/joins toggle one component.
_R2_1_SPECS = {
    "EV00": dict(ev_ret=False, ev_prompt=False, graph=True,  glossary=True,  values=True,  joins=True),
    "EV10": dict(ev_ret=True,  ev_prompt=False, graph=True,  glossary=True,  values=True,  joins=True),
    "EV01": dict(ev_ret=False, ev_prompt=True,  graph=True,  glossary=True,  values=True,  joins=True),
    "EV11": dict(ev_ret=True,  ev_prompt=True,  graph=True,  glossary=True,  values=True,  joins=True),
    "ABL_full":    dict(ev_ret=True, ev_prompt=True, graph=True,  glossary=True,  values=True,  joins=True),
    "ABL_nograph": dict(ev_ret=True, ev_prompt=True, graph=False, glossary=True,  values=True,  joins=True),
    "ABL_nogloss": dict(ev_ret=True, ev_prompt=True, graph=True,  glossary=False, values=True,  joins=True),
    "ABL_noval":   dict(ev_ret=True, ev_prompt=True, graph=True,  glossary=True,  values=False, joins=True),
    "ABL_nojoin":  dict(ev_ret=True, ev_prompt=True, graph=True,  glossary=True,  values=True,  joins=False),
}


def build_context(sg: SchemaGraph, question, condition, evidence=""):
    rq = _retrieval_query(question, evidence)   # used for embedding + anchors

    if condition == "C1":
        tables = set(sg.tables)
        cols = set(sg.columns.keys())
        join_conds = sorted({d["on"] for _, _, d in sg.graph.edges(data=True)
                             if d.get("etype") == "FK_TABLE" and "on" in d})
        ctx = render_context(sg, tables, cols, join_conds)

    elif condition == "C2":
        tables, cols = vector_topk(sg, rq)
        ctx = render_context(sg, tables, cols, None)

    elif condition == "C3":
        a_tables, _ = anchor_match(sg, rq)
        seed = a_tables if a_tables else set(list(sg.tables)[:1])
        tables = graph_expand(sg, seed)
        _, join_conds = shortest_join_path(sg, tables)
        cols = {c.qualified for t in tables for c in sg.table_columns(t)}
        ctx = render_context(sg, tables, cols, join_conds)

    elif condition == "C4":
        v_tables, v_cols = vector_topk(sg, rq)
        a_tables, a_cols = anchor_match(sg, rq)
        tables = graph_expand(sg, v_tables | a_tables)
        cols = v_cols | a_cols | {c.qualified for t in a_tables
                                  for c in sg.table_columns(t)}
        _, join_conds = shortest_join_path(sg, tables)
        ctx = render_context(sg, tables, cols, join_conds)

    elif condition.startswith("C5"):
        level = int(condition.split("_L")[-1]) if "_L" in condition else 3
        v_tables, v_cols = vector_topk(sg, rq)
        a_tables, a_cols = anchor_match(sg, rq)
        cand_tables = graph_expand(sg, v_tables | a_tables)
        cand_cols = v_cols | a_cols | {c.qualified for t in a_tables       # REVISION R2.3-ii: match C4 set
                                       for c in sg.table_columns(t)}
        tables, cols, join_conds = deterministic_prune(
            sg, cand_tables, cand_cols, rq, level=level)
        ctx = render_context(sg, tables, cols, join_conds)

    elif condition in _R2_1_SPECS:                                   # REVISION R2.1
        spec = _R2_1_SPECS[condition]
        q_ret = f"{question}\nHint: {evidence}" if (spec["ev_ret"] and evidence) else question
        v_tables, v_cols = vector_topk(sg, q_ret)
        am = anchor_match if spec["glossary"] else anchor_match_nogloss
        a_tables, a_cols = am(sg, q_ret)
        if spec["graph"]:
            tables = graph_expand(sg, v_tables | a_tables)
            _, join_conds = shortest_join_path(sg, tables)
        else:
            tables = set(v_tables) | set(a_tables)
            join_conds = []
        cols = v_cols | a_cols | {c.qualified for t in a_tables
                                  for c in sg.table_columns(t)}
        ctx = render_context(sg, tables, cols,
                             join_conds if spec["joins"] else None,
                             include_glossary=spec["glossary"],
                             include_samples=spec["values"],
                             include_joins=spec["joins"])
        if spec["ev_prompt"] and evidence:
            ctx = ctx + f"\n\n-- External knowledge (evidence):\n--   {evidence}"
        return ctx, {"tables": sorted(tables), "columns": sorted(cols)}

    elif condition in ("C1_EV0", "C1_EV1"):                          # REVISION R2.1
        tables = set(sg.tables); cols = set(sg.columns.keys())
        join_conds = sorted({d["on"] for _, _, d in sg.graph.edges(data=True)
                             if d.get("etype") == "FK_TABLE" and "on" in d})
        ctx = render_context(sg, tables, cols, join_conds)
        if condition == "C1_EV1" and evidence:
            ctx = ctx + f"\n\n-- External knowledge (evidence):\n--   {evidence}"
        return ctx, {"tables": sorted(tables), "columns": sorted(cols)}

    elif re.fullmatch(r"C4_k\d{4}", condition):                     # REVISION R3c (optional EX-at-k)
        kt, kc = int(condition[4:6]), int(condition[6:8])
        v_tables, v_cols = vector_topk(sg, rq, k_tables=kt, k_cols=kc)
        a_tables, a_cols = anchor_match(sg, rq)
        tables = graph_expand(sg, v_tables | a_tables)
        cols = v_cols | a_cols | {c.qualified for t in a_tables
                                  for c in sg.table_columns(t)}
        _, join_conds = shortest_join_path(sg, tables)
        ctx = render_context(sg, tables, cols, join_conds)
        if USE_EVIDENCE and evidence:
            ctx = ctx + f"\n\n-- External knowledge (evidence):\n--   {evidence}"
        return ctx, {"tables": sorted(tables), "columns": sorted(cols)}

    else:
        raise ValueError(f"Unknown condition: {condition}")

    # Append evidence to the context so the generator sees the same hint.
    if USE_EVIDENCE and evidence:
        ctx = ctx + f"\n\n-- External knowledge (evidence):\n--   {evidence}"

    return ctx, {"tables": sorted(tables), "columns": sorted(cols)}


# %% [markdown]
# ## Cell 10 - Build and persist contexts (passes evidence to the builder)

# %%
def count_tokens_approx(text):
    return max(1, int(len(text) / 4))


def build_all_contexts(dataset_key, limit=None):
    questions = json.loads(
        (DATA_DIR / dataset_key / "questions.json").read_text(encoding="utf-8"))
    if limit:
        questions = questions[:limit]

    schema_list = _load_schema_list(dataset_key)
    schema_by_db = {s["db_id"]: s for s in schema_list}
    graph_cache = {}

    rows, gold_rows = [], []
    for q in questions:
        db_id = q["db_id"]
        if db_id not in schema_by_db:
            continue
        if db_id not in graph_cache:
            graph_cache[db_id] = build_graph(schema_by_db[db_id])
        sg = graph_cache[db_id]
        evidence = q.get("evidence", "")           # <- pull evidence

        gold_rows.append({
            "dataset": dataset_key, "qid": q["qid"], "db_id": db_id,
            "gold_tables": "|".join(q.get("gold_tables", [])),
            "gold_columns": "|".join(q.get("gold_columns", [])),
        })

        for cond in BUILD_CONDITIONS:
            ctx, sel = build_context(sg, q["question"], cond, evidence=evidence)  # <- pass
            out_dir = CONTEXTS_DIR / dataset_key / cond
            out_dir.mkdir(parents=True, exist_ok=True)
            fpath = out_dir / f"{db_id}__{q['qid']}.txt"
            fpath.write_text(ctx, encoding="utf-8")

            rows.append({
                "dataset": dataset_key, "qid": q["qid"], "db_id": db_id,
                "condition": cond,
                "n_tables": len(sel["tables"]),
                "n_columns": len(sel["columns"]),
                "context_tokens_est": count_tokens_approx(ctx),
                "selected_tables": "|".join(sel["tables"]),
                "selected_columns": "|".join(sel["columns"]),
                "context_path": str(fpath.relative_to(PROJECT_ROOT)),
            })

    meta_df = pd.DataFrame(rows)
    meta_df.to_csv(CONTEXTS_DIR / f"{dataset_key}__context_meta.csv", index=False)

    gold_out = CONTEXTS_DIR / "schema_link_gold.csv"
    gold_df = pd.DataFrame(gold_rows)
    if gold_out.exists():
        prev = pd.read_csv(gold_out)
        prev = prev[prev["dataset"] != dataset_key]
        gold_df = pd.concat([prev, gold_df], ignore_index=True)
    gold_df.to_csv(gold_out, index=False)

    print(f"[{dataset_key}] built {len(meta_df)} contexts "
          f"({meta_df['qid'].nunique()} questions x {len(BUILD_CONDITIONS)} conditions).")
    return meta_df


# %% [markdown]
# ## Cell 11 - Run the builder

# %%

# ==========================================================================
# From 02_run_generation.ipynb — SQL validation + execution scoring
# ==========================================================================
import re
import sqlglot

def clean_sql(text):
    """Strip markdown fences / stray prose the model may add."""
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    m = re.search(r"(?is)\b(WITH|SELECT)\b", t)
    if m:
        t = t[m.start():]
    if ";" in t:
        t = t[: t.index(";") + 1]
    return t.strip()


def validate_sql(sql, dialect="sqlite"):
    try:
        return (sqlglot.parse_one(sql, read=dialect) is not None), ""
    except Exception as e:
        return False, str(e)


def _db_path(dataset_key, db_id):
    return DATA_DIR / dataset_key / "db" / f"{db_id}.sqlite"


def run_sql(dataset_key, db_id, sql, wall_limit_s=10.0):
    """
    Execute SQL with a real wall-clock limit. Returns (rows, error).
    rows is a list of tuples, or None on error/timeout.
    """
    path = _db_path(dataset_key, db_id)
    if not path.exists():
        return None, "DB not found"

    con = None
    deadline = time.time() + wall_limit_s

    def _guard():
        # Called periodically by SQLite; returning non-zero aborts the query.
        return 1 if time.time() > deadline else 0

    try:
        con = sqlite3.connect(str(path), timeout=5.0)
        con.text_factory = lambda b: b.decode("utf-8", "ignore")
        con.set_progress_handler(_guard, 1_000_000)
        cur = con.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        con.close()
        return rows, ""
    except Exception as e:
        if con is not None:
            try:
                con.close()
            except Exception:
                pass
        msg = str(e)
        if "interrupted" in msg.lower() or "aborted" in msg.lower():
            return None, f"SQL wall-clock timeout (> {wall_limit_s:.0f}s)"
        return None, msg


def _has_order_by(sql):
    return bool(re.search(r"(?is)\border\s+by\b", sql))


def compare_results(pred_rows, gold_rows, ordered):
    if pred_rows is None or gold_rows is None:
        return False
    def norm(rows):
        return [tuple("" if v is None else str(v) for v in r) for r in rows]
    p, g = norm(pred_rows), norm(gold_rows)
    if ordered:
        return p == g
    from collections import Counter
    return Counter(p) == Counter(g)


def execution_match(dataset_key, db_id, pred_sql, gold_sql):
    gold_rows, gold_err = run_sql(dataset_key, db_id, gold_sql)
    pred_rows, pred_err = run_sql(dataset_key, db_id, pred_sql)
    if gold_rows is None:
        return None, pred_err, gold_err     # unscoreable (e.g. DB missing)
    return compare_results(pred_rows, gold_rows, _has_order_by(gold_sql)), pred_err, gold_err


Core libraries imported.
Project directories ready.
Environment loaded. Model tiers:
  WEAK   -> gpt-4o-mini
  MID    -> gpt-5.4-mini
  STRONG -> gpt-5.4
  REASON -> gpt-6-luna
Conditions: ['C1', 'C2', 'C3', 'C4', 'C5']
Datasets   : ['D1', 'D2', 'D2_fin', 'D3']
Grayscale plotting style configured.
Condition styles: {'C1': '0.15', 'C2': '0.3', 'C3': '0.45', 'C4': '0.6', 'C5': '0.75'}
Live check skipped (set RUN_LIVE_CHECK=True to test credentials).
Setup complete. Config snapshot saved.
Notebook 01 dependencies ready.
Mode: FULL | datasets=['D1', 'D2', 'D2_fin'] | limit=None | evidence=True | k_tables=8 k_cols=30


In [2]:
# %% R2_4b - per-stage timing (graph build / retrieval / pruning / validation /
#            execution), with generation latency taken from the master run.
# Self-contained and offline: embeddings come from cache, execution runs on the
# local SQLite DBs, and generation latency is read from generation_results.csv.
import json, time, numpy as np, pandas as pd
from collections import defaultdict

SAMPLE_N = 60                 # questions per dataset for the timed stages
TIER     = "STRONG"
STAGE    = defaultdict(list)  # (stage, dataset) -> list[seconds]

master = pd.read_csv(TABLES_DIR / "generation_results.csv")

for ds in ["D2", "D1"]:
    schema_by_db = {s["db_id"]: s for s in _load_schema_list(ds)}
    questions = {q["qid"]: q for q in
                 json.loads((DATA_DIR / ds / "questions.json").read_text("utf-8"))}
    sub = (master[(master.dataset == ds) & (master.model_tier == TIER)
                  & (master.condition == "C4")]
           .drop_duplicates("qid").head(SAMPLE_N))

    # (1) graph build - timed once per unique db
    graph_cache = {}
    for db in {questions[r.qid]["db_id"] for r in sub.itertuples() if r.qid in questions}:
        if db not in schema_by_db:
            continue
        t = time.perf_counter(); graph_cache[db] = build_graph(schema_by_db[db])
        STAGE[("1_graph_build(per_db)", ds)].append(time.perf_counter() - t)

    for r in sub.itertuples():
        q = questions.get(r.qid)
        if not q or q["db_id"] not in graph_cache:
            continue
        sg, ev, db = graph_cache[q["db_id"]], q.get("evidence", ""), q["db_id"]
        # (2) retrieval + context assembly (C4)
        t = time.perf_counter(); ctx, sel = build_context(sg, q["question"], "C4", ev)
        STAGE[("2_retrieval+assemble(C4)", ds)].append(time.perf_counter() - t)
        # (3) retrieval + deterministic pruning (C5-L2); pruning cost = this - C4
        t = time.perf_counter(); build_context(sg, q["question"], "C5_L2", ev)
        STAGE[("3_retrieval+prune(C5_L2)", ds)].append(time.perf_counter() - t)
        # (4) SQL validation + (5) execution, on the recorded prediction
        pred = getattr(r, "pred_sql", "")
        pred = "" if (pred is None or (isinstance(pred, float))) else str(pred)
        if pred and pred.lower() != "nan":
            t = time.perf_counter(); validate_sql(pred)
            STAGE[("4_validation", ds)].append(time.perf_counter() - t)
            t = time.perf_counter()
            try:
                execution_match(ds, db, pred, q["gold_sql"])
            except Exception:
                pass
            STAGE[("5_execution", ds)].append(time.perf_counter() - t)

    # (6) generation latency - straight from the master run (no new API calls)
    glat = master[(master.dataset == ds) & (master.model_tier == TIER)
                  & (master.condition == "C4")]["latency_s"].dropna()
    STAGE[("6_generation(from_master)", ds)].extend(glat.tolist())
    print(f"{ds}: timed {len(sub)} questions over {len(graph_cache)} databases.")

rows = []
for (stage, ds), xs in STAGE.items():
    a = np.array(xs, float); a = a[~np.isnan(a)]
    if not len(a):
        continue
    rows.append({"dataset": ds, "stage": stage, "n": len(a),
                 "mean_ms": round(a.mean() * 1e3, 2),
                 "median_ms": round(np.median(a) * 1e3, 2),
                 "p95_ms": round(np.percentile(a, 95) * 1e3, 2),
                 "total_s": round(a.sum(), 3)})
stage_df = pd.DataFrame(rows).sort_values(["dataset", "stage"]).reset_index(drop=True)
display(stage_df)
save_table(stage_df, "NEW_stage_timings")
print("\nsaved NEW_stage_timings.csv")


D2: timed 60 questions over 2 databases.
D1: timed 60 questions over 2 databases.


,dataset,stage,n,mean_ms,median_ms,p95_ms,total_s
0,D1,1_graph_build(per_db),2,0.11,0.11,0.14,0.000
1,D1,2_retrieval+assemble(C4),60,11.77,12.55,13.27,0.706
2,D1,3_retrieval+prune(C5_L2),60,19.43,22.64,24.09,1.166
3,D1,4_validation,60,0.43,0.39,0.70,0.026
4,D1,5_execution,60,1.13,1.11,1.27,0.068
5,D1,6_generation(from_master),1034,114.41,0.00,1085.10,118.296
6,D2,1_graph_build(per_db),2,0.48,0.48,0.54,0.001
7,D2,2_retrieval+assemble(C4),60,47.07,44.72,58.43,2.824
8,D2,3_retrieval+prune(C5_L2),60,80.65,81.45,98.31,4.839
9,D2,4_validation,60,2.20,0.84,2.74,0.132


Saved table: NEW_stage_timings.csv, NEW_stage_timings.tex

saved NEW_stage_timings.csv


**Notes.** `graph_build` is a one-off per database (amortized over all its
questions); deterministic pruning's marginal cost is row 3 minus row 2.
Generation dominates wall-clock and is decode-bound (flat across conditions),
which is why the graph layer's benefit is context/cost efficiency rather than
speed. Send me `NEW_stage_timings.csv` and I will turn the current latency
paragraph into a per-stage table in sec. *End-to-End Latency and Cost*.
